# 4사분면 분석

모델별 Reference Similarity vs Stability 시각화

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/project_05

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_4quadrant_full(df):

    plt.figure(figsize=(9, 9))

    # ============================================================
    # 1. 축 정의 (너 실험 기준)
    # ============================================================

    df = df.copy()

    # 안정성 (낮을수록 나쁨 → 높을수록 좋게 변환)
    df["stability"] = 100 - df["hard_fail_rate"]

    # 유용성
    df["utility"] = df["avg_reference_similarity"]

    # ============================================================
    # 2. scatter plot (모델 + 학습 상태 구분)
    # ============================================================

    sns.scatterplot(
        data=df,
        x="stability",
        y="utility",
        hue="model_name",          # Qwen2.5 vs Qwen3.5
        style="model_type",        # base vs fine-tuned (LoRA 포함)
        s=140
    )

    # ============================================================
    # 3. 기준선 (사분면 나누기)
    # ============================================================

    x_mid = df["stability"].mean()
    y_mid = df["utility"].mean()

    plt.axvline(x_mid, linestyle="--", color="gray", linewidth=1)
    plt.axhline(y_mid, linestyle="--", color="gray", linewidth=1)

    # ============================================================
    # 4. 사분면 라벨
    # ============================================================

    plt.text(x_mid + 1, y_mid + 0.005, "Q1 (Best)", fontsize=10)
    plt.text(x_mid - 25, y_mid + 0.005, "Q2 (High utility / Low stability)", fontsize=10)
    plt.text(x_mid - 25, y_mid - 0.01, "Q3 (Worst)", fontsize=10)
    plt.text(x_mid + 1, y_mid - 0.01, "Q4 (Stable / Low utility)", fontsize=10)

    # ============================================================
    # 5. 점 라벨 (model + prompt 조합)
    # ============================================================

    for _, row in df.iterrows():

        label = f"{row['model_name']}\n{row['model_type']}\n{row['prompt_type']}"

        plt.text(
            row["stability"],
            row["utility"],
            label,
            fontsize=8,
            alpha=0.8
        )

    # ============================================================
    # 6. 스타일 (논문용)
    # ============================================================

    plt.title("4-Quadrant Performance Analysis (Utility vs Stability)", fontsize=14)
    plt.xlabel("Stability (1 - Hard Fail Rate)", fontsize=12)
    plt.ylabel("Utility (Reference Similarity)", fontsize=12)

    plt.grid(True, alpha=0.25)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd

CSV_FILES = [
    "/content/drive/MyDrive/project_05/eval_outputs_ab_v2_final.csv",
    "/content/drive/MyDrive/project_05/eval_outputs_qwen35_ab_final.csv",
]

dfs = []

for path in CSV_FILES:
    temp = pd.read_csv(path)
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print("total rows:", len(df))

In [ ]:
# 필수 feature 재생성 (중요)
df["korean_ratio"] = df["model_output"].apply(lambda x: sum(c.isalpha() for c in str(x)))
df["chinese_detected"] = df["model_output"].str.contains(r"[\u4e00-\u9fff]", na=False)
df["question_count"] = df["model_output"].apply(lambda x: str(x).count("?"))
df["single_question_pass"] = df["question_count"] == 1
df["prompt_leakage"] = df["model_output"].str.contains("rule|instruction|think", case=False, na=False)

df["hard_fail"] = df["chinese_detected"] | (df["question_count"] > 1)

In [ ]:
summary = (
    df.groupby(
        ["model_type", "model_name", "prompt_type", "persona"]
    )
    .agg(
        total=("model_output", "count"),
        avg_reference_similarity=("reference_similarity", "mean"),
        avg_korean_ratio=("korean_ratio", "mean"),
        chinese_rate=("chinese_detected", "mean"),
        avg_question_count=("question_count", "mean"),
        single_question_rate=("single_question_pass", "mean"),
        prompt_leakage_rate=("prompt_leakage", "mean"),
        style_pass_rate=("style_pass", "mean"),
        hard_fail_rate=("hard_fail", "mean"),
    )
    .reset_index()
)

for col in [
    "avg_reference_similarity",
    "avg_korean_ratio",
    "chinese_rate",
    "single_question_rate",
    "prompt_leakage_rate",
    "style_pass_rate",
    "hard_fail_rate",
]:
    summary[col] = (summary[col] * 100).round(2)

summary["avg_question_count"] = summary["avg_question_count"].round(2)

In [ ]:
print(df.shape)
df.head()

In [ ]:
plot_4quadrant_full(summary)